# Análisis Completo de MyAnimeList (MAL) Dataset
## Proyecto Integral: EDA, Visualizaciones, Animaciones y Machine Learning

**Dataset:** MyAnimeList - 19,931 animes  
**Objetivos:**
1. Análisis exploratorio exhaustivo
2. Visualizaciones estáticas e interactivas
3. Animaciones estilo race bar charts
4. Sistema de recomendación
5. Dashboard completo con Matplotlib y Seaborn
6. Preparación para web deployment

---

## 📚 ÍNDICE

### FASE 1: ANÁLISIS EXPLORATORIO
1. Configuración e importación de librerías
2. Carga y validación de datos
3. Estadísticas descriptivas
4. Análisis de valores faltantes
5. Visualizaciones básicas
6. Análisis de correlaciones

### FASE 2: VISUALIZACIONES AVANZADAS
7. Sistema de recomendación interactivo
8. Análisis por géneros
9. Análisis por estudios
10. Análisis temporal

### FASE 2.5: DASHBOARD Y SEABORN
11. Dashboard completo con Matplotlib (10 gráficas)
12. Análisis con Seaborn
13. Gráficas animadas (estilo Liga MX)

### BONUS: MACHINE LEARNING
14. Predicción de scores
15. Clustering de animes

---

# FASE 1: ANÁLISIS EXPLORATORIO

## 1. Configuración e Importación de Librerías

In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import warnings
import json
import os
from datetime import datetime

# Visualización
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from matplotlib import cm
from matplotlib.patches import Rectangle
import seaborn as sns

# Interactivos
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Matplotlib config
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Seaborn config
sns.set_palette('husl')

%matplotlib inline

print("✓ Librerías importadas correctamente")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Crear carpetas para organizar outputs
folders = ['graficas_individuales', 'graficas_interactivas', 'animaciones', 'datos_procesados']
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    
print("✓ Carpetas creadas:")
for folder in folders:
    print(f"  - {folder}/")

## 2. Carga y Validación de Datos

In [ ]:
# Cargar dataset
df = pd.read_csv('mal_anime.csv')

print("="*70)
print("DATASET CARGADO: MyAnimeList (MAL)")
print("="*70)
print(f"\n📊 Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"\n📋 Columnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Información general del dataset
print("\n" + "="*70)
print("INFORMACIÓN GENERAL DEL DATASET")
print("="*70)
df.info()

In [ ]:
# Primeras filas
print("\n" + "="*70)
print("MUESTRA DE DATOS (Primeras 5 filas)")
print("="*70)
display(df.head())

In [ ]:
# Últimas filas
print("\n" + "="*70)
print("MUESTRA DE DATOS (Últimas 5 filas)")
print("="*70)
display(df.tail())

## 3. Estadísticas Descriptivas

In [ ]:
print("="*70)
print("ESTADÍSTICAS DESCRIPTIVAS - VARIABLES NUMÉRICAS")
print("="*70)

# Seleccionar solo columnas numéricas relevantes
numeric_cols = ['Score', 'Members', 'Favorites', 'Released_Year']
numeric_data = df[numeric_cols].copy()

display(numeric_data.describe().round(2))

In [ ]:
print("\n" + "="*70)
print("ESTADÍSTICAS DESCRIPTIVAS - VARIABLES CATEGÓRICAS")
print("="*70)

categorical_cols = ['Type', 'Status', 'Source', 'Rating', 'Demographic']

for col in categorical_cols:
    if col in df.columns:
        print(f"\n{'='*70}")
        print(f"{col.upper()}")
        print(f"{'='*70}")
        print(f"  Valores únicos: {df[col].nunique()}")
        print(f"  Valores faltantes: {df[col].isna().sum()} ({(df[col].isna().sum()/len(df)*100):.1f}%)")
        print(f"\n  Top 10:")
        value_counts = df[col].value_counts().head(10)
        for idx, (val, count) in enumerate(value_counts.items(), 1):
            pct = (count / len(df)) * 100
            print(f"    {idx:2d}. {str(val):30s}: {count:6,} ({pct:5.2f}%)")

### 3.1 Análisis Especial: Géneros y Temas

In [ ]:
# Los géneros y temas vienen separados por comas, necesitamos procesarlos
def extract_and_count(series, column_name):
    """
    Extrae valores separados por comas y cuenta frecuencias
    """
    all_items = []
    for item in series.dropna():
        if isinstance(item, str):
            items = [x.strip() for x in item.split(',')]
            all_items.extend(items)
    
    from collections import Counter
    counts = Counter(all_items)
    
    print(f"\n{'='*70}")
    print(f"{column_name.upper()}")
    print(f"{'='*70}")
    print(f"  Total de {column_name.lower()} únicos: {len(counts)}")
    print(f"\n  Top 15:")
    
    for idx, (item, count) in enumerate(counts.most_common(15), 1):
        pct = (count / len(series)) * 100
        print(f"    {idx:2d}. {item:30s}: {count:6,} ({pct:5.2f}%)")
    
    return counts

# Analizar géneros
genres_counts = extract_and_count(df['Genres'], 'Géneros')

# Analizar temas
themes_counts = extract_and_count(df['Themes'], 'Temas')

# Analizar estudios
studios_counts = extract_and_count(df['Studios'], 'Estudios')

## 4. Análisis de Valores Faltantes

In [ ]:
print("="*70)
print("ANÁLISIS DE VALORES FALTANTES")
print("="*70)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Valores Faltantes': missing.values,
    'Porcentaje (%)': missing_pct.values
})

missing_df = missing_df[missing_df['Valores Faltantes'] > 0].sort_values(
    'Porcentaje (%)', ascending=False
)

if len(missing_df) == 0:
    print("\n  ✓ No hay valores faltantes en el dataset")
else:
    print(f"\n  ⚠️  Se encontraron valores faltantes en {len(missing_df)} columnas:\n")
    display(missing_df)

total_missing = missing.sum()
total_cells = df.shape[0] * df.shape[1]
print(f"\n  Total de valores faltantes: {total_missing:,}")
print(f"  Porcentaje total: {(total_missing/total_cells)*100:.2f}%")

## 5. Visualizaciones Básicas

### 5.1 Distribución de Scores

In [ ]:
# Distribución de scores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma
axes[0].hist(df['Score'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='#FF6B6B')
axes[0].axvline(df['Score'].mean(), color='blue', linestyle='--', linewidth=2, 
               label=f'Media: {df["Score"].mean():.2f}')
axes[0].axvline(df['Score'].median(), color='green', linestyle='-.', linewidth=2,
               label=f'Mediana: {df["Score"].median():.2f}')
axes[0].set_xlabel('Score', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frecuencia', fontsize=12, fontweight='bold')
axes[0].set_title('Distribución de Scores de Anime', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Boxplot
bp = axes[1].boxplot(df['Score'].dropna(), vert=True, patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][0].set_alpha(0.7)
axes[1].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[1].set_title('Boxplot de Scores', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('graficas_individuales/01_distribucion_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 Estadísticas de Score:")
print(f"  Media: {df['Score'].mean():.2f}")
print(f"  Mediana: {df['Score'].median():.2f}")
print(f"  Desv. Est.: {df['Score'].std():.2f}")
print(f"  Rango: {df['Score'].min():.2f} - {df['Score'].max():.2f}")

### 5.2 Top Animes por Score

In [ ]:
# Top 20 animes por score
top_by_score = df.nlargest(20, 'Score')[['title', 'Score', 'Type', 'Genres', 'Members']]

print("="*70)
print("TOP 20 ANIMES POR SCORE")
print("="*70)
display(top_by_score)

# Visualización
fig, ax = plt.subplots(figsize=(12, 8))

top_20_scores = df.nlargest(20, 'Score')
colors = plt.cm.viridis(np.linspace(0, 1, 20))

bars = ax.barh(range(20), top_20_scores['Score'].values, color=colors, edgecolor='black', linewidth=1.5)
ax.set_yticks(range(20))
ax.set_yticklabels(top_20_scores['title'].values, fontsize=10)
ax.set_xlabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Animes por Score', fontsize=14, fontweight='bold', pad=20)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Añadir valores
for i, (bar, score) in enumerate(zip(bars, top_20_scores['Score'].values)):
    ax.text(score + 0.05, i, f'{score:.2f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('graficas_individuales/02_top20_scores.png', dpi=300, bbox_inches='tight')
plt.show()

### 5.3 Distribución por Tipo

In [ ]:
# Distribución por tipo
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart
type_counts = df['Type'].value_counts().head(8)
colors_pie = plt.cm.Set3(range(len(type_counts)))
explode = [0.05 if i == 0 else 0 for i in range(len(type_counts))]

axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
           colors=colors_pie, explode=explode, shadow=True, startangle=90)
axes[0].set_title('Distribución por Tipo de Anime', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[1].bar(range(len(type_counts)), type_counts.values, color=colors_pie, edgecolor='black', linewidth=1.5)
axes[1].set_xticks(range(len(type_counts)))
axes[1].set_xticklabels(type_counts.index, rotation=45, ha='right')
axes[1].set_ylabel('Cantidad', fontsize=12, fontweight='bold')
axes[1].set_title('Cantidad por Tipo', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, type_counts.values):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('graficas_individuales/03_distribucion_tipo.png', dpi=300, bbox_inches='tight')
plt.show()

### 5.4 Análisis Temporal

In [ ]:
# Animes por año
df_with_year = df[df['Released_Year'].notna()].copy()
df_with_year = df_with_year[df_with_year['Released_Year'] >= 1960]  # Filtrar años válidos

animes_by_year = df_with_year.groupby('Released_Year').size().sort_index()  # Ordenar por año

fig, ax = plt.subplots(figsize=(16, 6))

ax.fill_between(animes_by_year.index, 0, animes_by_year.values, alpha=0.4, color='#3498db')
ax.plot(animes_by_year.index, animes_by_year.values, color='#e74c3c', linewidth=2.5, marker='o', markersize=3)

ax.set_xlabel('Año', fontsize=12, fontweight='bold')
ax.set_ylabel('Cantidad de Animes', fontsize=12, fontweight='bold')
ax.set_title('Evolución de Animes Lanzados por Año', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim(animes_by_year.index.min(), animes_by_year.index.max())  # Asegurar orden correcto

plt.tight_layout()
plt.savefig('graficas_individuales/04_animes_por_año.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 Estadísticas temporales:")
print(f"  Año con más lanzamientos: {animes_by_year.idxmax():.0f} ({animes_by_year.max()} animes)")
print(f"  Promedio anual: {animes_by_year.mean():.0f} animes")

## 6. Análisis de Correlaciones

In [ ]:
# Preparar datos numéricos para correlación
# Convertir Members y Favorites a numéricos si están como string
df['Members_num'] = pd.to_numeric(df['Members'].astype(str).str.replace(',', ''), errors='coerce')
df['Favorites_num'] = pd.to_numeric(df['Favorites'].astype(str).str.replace(',', ''), errors='coerce')

# Seleccionar variables para correlación
corr_vars = ['Score', 'Members_num', 'Favorites_num', 'Released_Year']
corr_data = df[corr_vars].dropna()

# Calcular correlación
corr_matrix = corr_data.corr()

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap con Seaborn
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
           square=True, linewidths=1, cbar_kws={"shrink": 0.8},
           vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Matriz de Correlación (Seaborn)', fontsize=14, fontweight='bold')

# Heatmap con imshow
im = axes[1].imshow(corr_matrix, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr_matrix.columns)))
axes[1].set_yticks(range(len(corr_matrix.columns)))
axes[1].set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
axes[1].set_yticklabels(corr_matrix.columns)
axes[1].set_title('Matriz de Correlación (imshow)', fontsize=14, fontweight='bold')

# Añadir valores
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        axes[1].text(j, i, f'{corr_matrix.iloc[i, j]:.3f}',
                    ha='center', va='center', fontsize=10, fontweight='bold')

plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('graficas_individuales/05_correlaciones.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Correlaciones con Score:")
score_corr = corr_matrix['Score'].sort_values(ascending=False)
for var, corr in score_corr.items():
    if var != 'Score':
        print(f"  {var:20s}: {corr:6.3f}")

---
# FASE 2: VISUALIZACIONES AVANZADAS E INTERACTIVAS

## 7. Sistema de Recomendación Interactivo

In [ ]:
# Preparar datos para recomendación por género
def get_top_animes_by_genre(genre, df, top_n=10):
    """
    Obtiene los top N animes para un género específico
    """
    # Filtrar animes que contengan el género
    genre_animes = df[df['Genres'].str.contains(genre, na=False, case=False)].copy()
    
    # Ordenar por score y luego por members
    genre_animes = genre_animes.sort_values(['Score', 'Members_num'], ascending=[False, False])
    
    return genre_animes.head(top_n)[['title', 'Score', 'Type', 'Genres', 'Members_num']]

# Obtener todos los géneros únicos
all_genres = set()
for genres in df['Genres'].dropna():
    if isinstance(genres, str):
        for genre in genres.split(','):
            all_genres.add(genre.strip())

all_genres = sorted(list(all_genres))

print(f"Total de géneros únicos: {len(all_genres)}")
print(f"\nGéneros disponibles: {', '.join(all_genres[:20])}...")

In [ ]:
# Crear visualización interactiva con Plotly
# Preparar datos para cada género (top 5 géneros más comunes)
from collections import Counter

all_genres_list = []
for genres in df['Genres'].dropna():
    if isinstance(genres, str):
        all_genres_list.extend([g.strip() for g in genres.split(',')])

genre_counts = Counter(all_genres_list)
top_genres = [g for g, _ in genre_counts.most_common(20)]

# Crear figura con dropdown
fig = go.Figure()

for i, genre in enumerate(top_genres):
    top_animes = get_top_animes_by_genre(genre, df, top_n=10)
    
    fig.add_trace(
        go.Bar(
            name=genre,
            x=top_animes['title'].values,
            y=top_animes['Score'].values,
            visible=(i == 0),
            marker=dict(
                color=top_animes['Score'].values,
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title="Score")
            ),
            text=top_animes['Score'].values,
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Score: %{y:.2f}<extra></extra>'
        )
    )

# Crear botones para dropdown
buttons = []
for i, genre in enumerate(top_genres):
    visible = [False] * len(top_genres)
    visible[i] = True
    
    buttons.append(
        dict(
            label=genre,
            method='update',
            args=[
                {'visible': visible},
                {'title': f'Top 10 Animes: {genre}'}
            ]
        )
    )

fig.update_layout(
    title=f'Top 10 Animes: {top_genres[0]}',
    xaxis_title='Anime',
    yaxis_title='Score',
    height=600,
    showlegend=False,
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.01,
            xanchor='left',
            y=1.15,
            yanchor='top',
            bgcolor='lightgray',
            bordercolor='gray',
            font=dict(size=11)
        )
    ],
    annotations=[
        dict(
            text='Selecciona un género:',
            showarrow=False,
            x=0.01,
            y=1.18,
            xref='paper',
            yref='paper',
            align='left',
            font=dict(size=13, color='black')
        )
    ],
    xaxis_tickangle=-45
)

fig.write_html('graficas_interactivas/sistema_recomendacion_generos.html')
print("✓ Sistema de recomendación guardado: graficas_interactivas/sistema_recomendacion_generos.html")
fig.show()

## 8. Análisis por Géneros

In [ ]:
# Top 15 géneros más comunes
top_15_genres = [g for g, _ in genre_counts.most_common(15)]
top_15_counts = [genre_counts[g] for g in top_15_genres]

fig, ax = plt.subplots(figsize=(12, 8))

colors_bar = plt.cm.Spectral(np.linspace(0, 1, 15))
bars = ax.barh(range(15), top_15_counts, color=colors_bar, edgecolor='black', linewidth=1.5)

ax.set_yticks(range(15))
ax.set_yticklabels(top_15_genres, fontsize=11)
ax.set_xlabel('Cantidad de Animes', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Géneros Más Populares', fontsize=14, fontweight='bold', pad=20)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

for i, (bar, val) in enumerate(zip(bars, top_15_counts)):
    ax.text(val + 50, i, f'{val:,}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('graficas_individuales/06_top_generos.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Análisis por Estudios

In [ ]:
# Top 15 estudios
top_15_studios = [s for s, _ in studios_counts.most_common(15)]
top_15_studio_counts = [studios_counts[s] for s in top_15_studios]

fig, ax = plt.subplots(figsize=(12, 8))

colors_bar = plt.cm.plasma(np.linspace(0, 1, 15))
bars = ax.barh(range(15), top_15_studio_counts, color=colors_bar, edgecolor='black', linewidth=1.5)

ax.set_yticks(range(15))
ax.set_yticklabels(top_15_studios, fontsize=11)
ax.set_xlabel('Cantidad de Animes', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Estudios con Más Producciones', fontsize=14, fontweight='bold', pad=20)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

for i, (bar, val) in enumerate(zip(bars, top_15_studio_counts)):
    ax.text(val + 10, i, f'{val:,}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('graficas_individuales/07_top_estudios.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Análisis por Temporada

In [ ]:
# Distribución por temporada
season_counts = df['Released_Season'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart
colors_season = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
axes[0].pie(season_counts.values, labels=season_counts.index, autopct='%1.1f%%',
           colors=colors_season, shadow=True, startangle=90)
axes[0].set_title('Distribución por Temporada', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[1].bar(season_counts.index, season_counts.values, color=colors_season, 
                   edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Cantidad', fontsize=12, fontweight='bold')
axes[1].set_title('Animes por Temporada', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, season_counts.values):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('graficas_individuales/08_distribucion_temporada.png', dpi=300, bbox_inches='tight')
plt.show()

---
# FASE 2.5: DASHBOARD COMPLETO Y ANIMACIONES

## 11. Dashboard Completo con Matplotlib

In [ ]:
# Crear dashboard con 10 gráficas
fig = plt.figure(figsize=(24, 20))
gs = GridSpec(4, 3, figure=fig, hspace=0.4, wspace=0.3)

fig.suptitle('Dashboard Completo: Análisis de MyAnimeList', 
             fontsize=20, fontweight='bold', y=0.995)

# GRÁFICA 1: Scatter Score vs Members
ax1 = fig.add_subplot(gs[0, 0])
scatter_data = df[['Score', 'Members_num', 'Favorites_num']].dropna()
scatter = ax1.scatter(scatter_data['Members_num'], scatter_data['Score'],
                     c=scatter_data['Favorites_num'], s=50, alpha=0.6,
                     cmap='viridis', edgecolors='black', linewidth=0.5)
ax1.set_xlabel('Members', fontsize=11, fontweight='bold')
ax1.set_ylabel('Score', fontsize=11, fontweight='bold')
ax1.set_title('1. Scatter: Members vs Score\n(color=Favorites)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')
plt.colorbar(scatter, ax=ax1, label='Favorites')

# GRÁFICA 2: Pie Chart - Tipos
ax2 = fig.add_subplot(gs[0, 1])
type_counts_dash = df['Type'].value_counts().head(6)
colors_pie = plt.cm.Set3(range(len(type_counts_dash)))
ax2.pie(type_counts_dash.values, labels=type_counts_dash.index,
        autopct='%1.1f%%', colors=colors_pie, shadow=True, startangle=90)
ax2.set_title('2. Pie: Distribución por Tipo', fontsize=12, fontweight='bold')

# GRÁFICA 3: Line Plot - Evolución temporal
ax3 = fig.add_subplot(gs[0, 2])
yearly_avg_score = df_with_year.groupby('Released_Year')['Score'].mean()
yearly_count = df_with_year.groupby('Released_Year').size()
ax3.plot(yearly_avg_score.index, yearly_avg_score.values, 
         marker='o', linewidth=2.5, markersize=4, label='Score Promedio', color='#FF6B6B')
ax3_twin = ax3.twinx()
ax3_twin.fill_between(yearly_count.index, 0, yearly_count.values, 
                       alpha=0.3, color='#3498db', label='Cantidad')
ax3.set_xlabel('Año', fontsize=11, fontweight='bold')
ax3.set_ylabel('Score Promedio', fontsize=11, fontweight='bold', color='#FF6B6B')
ax3_twin.set_ylabel('Cantidad', fontsize=11, fontweight='bold', color='#3498db')
ax3.set_title('3. Line: Evolución Temporal', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# GRÁFICA 4: Heatmap - Correlaciones
ax4 = fig.add_subplot(gs[1, 0])
im = ax4.imshow(corr_matrix, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
ax4.set_xticks(range(len(corr_matrix.columns)))
ax4.set_yticks(range(len(corr_matrix.columns)))
ax4.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=9)
ax4.set_yticklabels(corr_matrix.columns, fontsize=9)
ax4.set_title('4. Heatmap: Correlaciones', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax4)

# GRÁFICA 5: Barras Horizontales - Top Géneros
ax5 = fig.add_subplot(gs[1, 1])
top_10_genres = [g for g, _ in genre_counts.most_common(10)]
top_10_counts = [genre_counts[g] for g in top_10_genres]
colors_bar = plt.cm.Spectral(np.linspace(0, 1, 10))
bars = ax5.barh(range(10), top_10_counts, color=colors_bar, edgecolor='black', linewidth=1)
ax5.set_yticks(range(10))
ax5.set_yticklabels(top_10_genres, fontsize=9)
ax5.set_xlabel('Cantidad', fontsize=11, fontweight='bold')
ax5.set_title('5. Barras H: Top 10 Géneros', fontsize=12, fontweight='bold')
ax5.invert_yaxis()
ax5.grid(True, alpha=0.3, axis='x')

# GRÁFICA 6: Histograma - Distribución de Scores
ax6 = fig.add_subplot(gs[1, 2])
n, bins, patches = ax6.hist(df['Score'].dropna(), bins=40, edgecolor='black', linewidth=0.8)
cm_hist = plt.cm.plasma
norm = plt.Normalize(vmin=n.min(), vmax=n.max())
for patch, height in zip(patches, n):
    patch.set_facecolor(cm_hist(norm(height)))
ax6.axvline(df['Score'].mean(), color='red', linestyle='--', linewidth=2)
ax6.set_xlabel('Score', fontsize=11, fontweight='bold')
ax6.set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
ax6.set_title('6. Histograma: Scores', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='y')

# GRÁFICA 7: Boxplot - Scores por Tipo
ax7 = fig.add_subplot(gs[2, 0])
top_types = df['Type'].value_counts().head(5).index
data_boxplot = [df[df['Type'] == t]['Score'].dropna().values for t in top_types]
bp = ax7.boxplot(data_boxplot, labels=top_types, patch_artist=True)
colors_box = plt.cm.Set2(range(5))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax7.set_ylabel('Score', fontsize=11, fontweight='bold')
ax7.set_title('7. Boxplot: Score por Tipo', fontsize=12, fontweight='bold')
ax7.set_xticklabels(top_types, rotation=45, ha='right', fontsize=9)
ax7.grid(True, alpha=0.3, axis='y')

# GRÁFICA 8: Violin Plot - Scores por Rating
ax8 = fig.add_subplot(gs[2, 1])
top_ratings = df['Rating'].value_counts().head(4).index
data_violin = [df[df['Rating'] == r]['Score'].dropna().values for r in top_ratings]
parts = ax8.violinplot(data_violin, positions=range(4), widths=0.7,
                       showmeans=True, showextrema=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(plt.cm.Set2(i))
    pc.set_alpha(0.7)
ax8.set_xticks(range(4))
ax8.set_xticklabels(top_ratings, rotation=45, ha='right', fontsize=9)
ax8.set_ylabel('Score', fontsize=11, fontweight='bold')
ax8.set_title('8. Violin: Score por Rating', fontsize=12, fontweight='bold')
ax8.grid(True, alpha=0.3, axis='y')

# GRÁFICA 9: Barras - Top Estudios
ax9 = fig.add_subplot(gs[2, 2])
top_10_studios = [s for s, _ in studios_counts.most_common(10)]
top_10_studio_counts = [studios_counts[s] for s in top_10_studios]
bars = ax9.bar(range(10), top_10_studio_counts, color=plt.cm.viridis(np.linspace(0, 1, 10)),
              edgecolor='black', linewidth=1)
ax9.set_xticks(range(10))
ax9.set_xticklabels(top_10_studios, rotation=90, ha='right', fontsize=8)
ax9.set_ylabel('Cantidad', fontsize=11, fontweight='bold')
ax9.set_title('9. Barras: Top 10 Estudios', fontsize=12, fontweight='bold')
ax9.grid(True, alpha=0.3, axis='y')

# GRÁFICA 10: Área - Distribución temporal
ax10 = fig.add_subplot(gs[3, :])
x_vals = animes_by_year.index
ax10.fill_between(x_vals, 0, animes_by_year.values, alpha=0.5, color='#3498db')
ax10.plot(x_vals, animes_by_year.values, color='#e74c3c', linewidth=2.5)
ax10.set_xlabel('Año', fontsize=12, fontweight='bold')
ax10.set_ylabel('Cantidad de Animes', fontsize=12, fontweight='bold')
ax10.set_title('10. Área: Evolución Histórica de Lanzamientos', fontsize=12, fontweight='bold')
ax10.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('graficas_individuales/dashboard_completo.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Dashboard completo generado")

## 12. Análisis Avanzado con Seaborn

### 12.1 Pairplot

In [ ]:
# Crear pairplot con variables clave
pairplot_data = df[['Score', 'Members_num', 'Favorites_num']].dropna().sample(min(2000, len(df)))

pairplot_fig = sns.pairplot(pairplot_data, diag_kind='kde', height=3,
                            plot_kws={'alpha': 0.6, 's': 30, 'edgecolor': 'black', 'linewidth': 0.5},
                            diag_kws={'linewidth': 2})

pairplot_fig.fig.suptitle('Pairplot: Relaciones entre Variables Clave (Muestra de 2000)', 
                          fontsize=16, fontweight='bold', y=1.001)

plt.savefig('graficas_individuales/seaborn_pairplot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Pairplot generado")

### 12.2 Jointplot

In [ ]:
# Jointplot Score vs Members
joint_data = df[['Score', 'Members_num']].dropna().sample(min(2000, len(df)))

joint_fig = sns.jointplot(data=joint_data, x='Members_num', y='Score',
                          kind='reg', color='#3498db', height=8,
                          marginal_kws={'bins': 30, 'color': '#e74c3c'},
                          joint_kws={'scatter_kws': {'alpha': 0.5, 's': 30, 'edgecolors': 'black', 'linewidths': 0.5},
                                    'line_kws': {'color': 'red', 'linewidth': 2.5}})

joint_fig.fig.suptitle('Jointplot: Members vs Score (Muestra)', fontsize=14, fontweight='bold', y=1.02)
joint_fig.ax_joint.set_xlabel('Members', fontsize=12, fontweight='bold')
joint_fig.ax_joint.set_ylabel('Score', fontsize=12, fontweight='bold')
joint_fig.ax_joint.set_xscale('log')

corr_val = joint_data['Members_num'].corr(joint_data['Score'])
joint_fig.ax_joint.text(0.05, 0.95, f'Correlación: {corr_val:.3f}',
                       transform=joint_fig.ax_joint.transAxes,
                       fontsize=12, fontweight='bold',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.savefig('graficas_individuales/seaborn_jointplot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Jointplot generado")

### 12.3 Violin Plots

In [ ]:
# Violin plots por tipo y temporada
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Por tipo
top_5_types = df['Type'].value_counts().head(5).index
df_top_types = df[df['Type'].isin(top_5_types)]
sns.violinplot(data=df_top_types, x='Type', y='Score', palette='muted',
              inner='box', ax=axes[0], linewidth=1.5)
axes[0].set_title('Distribución de Score por Tipo', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tipo', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Por temporada
df_with_season = df[df['Released_Season'].notna()]
sns.violinplot(data=df_with_season, x='Released_Season', y='Score', palette='Set2',
              inner='box', ax=axes[1], linewidth=1.5)
axes[1].set_title('Distribución de Score por Temporada', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Temporada', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('graficas_individuales/seaborn_violinplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Violin plots generados")

## 13. Animaciones Estilo Liga MX (Race Bar Charts)

### 13.1 Animación: Ranking de Géneros a lo Largo del Tiempo

In [ ]:
# Preparar datos para animación
# Contar animes por género y año
df_with_year_clean = df[(df['Released_Year'].notna()) & (df['Released_Year'] >= 1980)].copy()
df_with_year_clean['Released_Year'] = df_with_year_clean['Released_Year'].astype(int)

# Extraer géneros individuales
genre_year_data = []

for _, row in df_with_year_clean.iterrows():
    year = row['Released_Year']
    if pd.notna(row['Genres']) and isinstance(row['Genres'], str):
        genres = [g.strip() for g in row['Genres'].split(',')]
        for genre in genres:
            genre_year_data.append({'year': year, 'genre': genre})

genre_year_df = pd.DataFrame(genre_year_data)

# Contar acumulativamente
years = sorted(genre_year_df['year'].unique())
all_genres_in_data = genre_year_df['genre'].unique()

# Seleccionar top 15 géneros más comunes
top_genres_for_anim = [g for g, _ in genre_counts.most_common(15)]

# Crear datos acumulativos
cumulative_data = []

for year in years:
    year_data = genre_year_df[genre_year_df['year'] <= year]
    genre_counts_year = year_data['genre'].value_counts()
    
    for genre in top_genres_for_anim:
        count = genre_counts_year.get(genre, 0)
        cumulative_data.append({
            'year': year,
            'genre': genre,
            'count': count
        })

animation_df = pd.DataFrame(cumulative_data)

print(f"✓ Datos preparados para animación")
print(f"  Años: {len(years)} ({years[0]}-{years[-1]})")
print(f"  Géneros: {len(top_genres_for_anim)}")
print(f"  Total frames: {len(years)}")

In [ ]:
# Crear animación con Plotly (más eficiente)
# Tomar cada 5 años para hacer la animación más rápida
years_sampled = [y for y in years if y % 5 == 0 or y == years[-1]]
animation_df_sampled = animation_df[animation_df['year'].isin(years_sampled)]

fig_anim = px.bar(
    animation_df_sampled,
    x='count',
    y='genre',
    animation_frame='year',
    color='count',
    color_continuous_scale='Viridis',
    orientation='h',
    title='Evolución de Géneros de Anime (1980-2024)',
    labels={'count': 'Cantidad Acumulada', 'genre': 'Género'},
    range_x=[0, animation_df_sampled['count'].max() * 1.1]
)

fig_anim.update_layout(
    height=700,
    xaxis_title='Cantidad Acumulada de Animes',
    yaxis_title='',
    yaxis={'categoryorder': 'total ascending'},
    font=dict(size=12)
)

fig_anim.write_html('animaciones/race_chart_generos.html')
print("✓ Animación guardada: animaciones/race_chart_generos.html")
fig_anim.show()

### 13.2 Animación: Evolución de Scores Promedio

In [ ]:
# Animación de scores promedio por género a lo largo del tiempo
# Preparar datos
genre_score_data = []

for _, row in df_with_year_clean.iterrows():
    year = row['Released_Year']
    score = row['Score']
    if pd.notna(row['Genres']) and isinstance(row['Genres'], str) and pd.notna(score):
        genres = [g.strip() for g in row['Genres'].split(',')]
        for genre in genres:
            if genre in top_genres_for_anim:
                genre_score_data.append({'year': year, 'genre': genre, 'score': score})

genre_score_df = pd.DataFrame(genre_score_data)

# Calcular scores promedio acumulativos
avg_scores_data = []

for year in years_sampled:
    year_data = genre_score_df[genre_score_df['year'] <= year]
    avg_scores = year_data.groupby('genre')['score'].mean()
    
    for genre in top_genres_for_anim:
        if genre in avg_scores.index:
            avg_scores_data.append({
                'year': year,
                'genre': genre,
                'avg_score': avg_scores[genre]
            })

avg_scores_anim_df = pd.DataFrame(avg_scores_data)

# Crear animación
fig_scores = px.bar(
    avg_scores_anim_df,
    x='avg_score',
    y='genre',
    animation_frame='year',
    color='avg_score',
    color_continuous_scale='RdYlGn',
    orientation='h',
    title='Evolución de Scores Promedio por Género',
    labels={'avg_score': 'Score Promedio', 'genre': 'Género'},
    range_x=[0, 10]
)

fig_scores.update_layout(
    height=700,
    xaxis_title='Score Promedio',
    yaxis_title='',
    yaxis={'categoryorder': 'total ascending'},
    font=dict(size=12)
)

fig_scores.write_html('animaciones/evolucion_scores_generos.html')
print("✓ Animación guardada: animaciones/evolucion_scores_generos.html")
fig_scores.show()

### 13.3 Animación: Scatter Evolutivo (Members vs Score)

In [ ]:
# Scatter animado mostrando evolución temporal
scatter_anim_data = df_with_year_clean[
    (df_with_year_clean['Score'].notna()) & 
    (df_with_year_clean['Members_num'].notna()) &
    (df_with_year_clean['Favorites_num'].notna()) &
    (df_with_year_clean['Members_num'] > 0) &
    (df_with_year_clean['Favorites_num'] > 0)
].copy()

# Convertir año a int para la animación
scatter_anim_data['Released_Year'] = scatter_anim_data['Released_Year'].astype(int)

# Tomar muestra para performance
scatter_anim_data = scatter_anim_data.sample(min(3000, len(scatter_anim_data)))

# Crear animación solo con años que tengan suficientes datos
years_with_data = scatter_anim_data.groupby('Released_Year').size()
years_to_include = years_with_data[years_with_data >= 10].index.tolist()
scatter_anim_data = scatter_anim_data[scatter_anim_data['Released_Year'].isin(years_to_include)]

if len(scatter_anim_data) > 0:
    fig_scatter = px.scatter(
        scatter_anim_data,
        x='Members_num',
        y='Score',
        animation_frame='Released_Year',
        color='Type',
        size='Favorites_num',
        hover_data=['title', 'Type'],
        log_x=True,
        range_y=[0, 10],
        title='Evolución: Members vs Score por Año',
        labels={'Members_num': 'Members (log scale)', 'Score': 'Score'}
    )
    
    fig_scatter.update_layout(
        height=600,
        xaxis_title='Members (escala logarítmica)',
        yaxis_title='Score'
    )
    
    fig_scatter.write_html('animaciones/scatter_evolutivo.html')
    print("✓ Animación guardada: animaciones/scatter_evolutivo.html")
    fig_scatter.show()
else:
    print("⚠️  No hay suficientes datos para la animación scatter")

---
# BONUS: MACHINE LEARNING

## 14. Predicción de Scores (Opcional)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Preparar features
ml_data = df[['Score', 'Members_num', 'Favorites_num', 'Released_Year', 'Type']].dropna().copy()

# Encodear Type
ml_data = pd.get_dummies(ml_data, columns=['Type'], prefix='Type')

# Separar X y y
X = ml_data.drop('Score', axis=1)
y = ml_data['Score']

print(f"Dataset para ML: {len(X):,} registros con {X.shape[1]} features")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nEntrenando modelo Random Forest...")
# Entrenar modelo
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predecir
y_pred = rf_model.predict(X_test)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("="*70)
print("RESULTADOS DEL MODELO DE PREDICCIÓN DE SCORES")
print("="*70)
print(f"\nRMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Features más importantes:")
display(feature_importance.head(10))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predicciones vs Real
axes[0].scatter(y_test, y_pred, alpha=0.5, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
            'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Score Real', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score Predicho', fontsize=12, fontweight='bold')
axes[0].set_title(f'Predicción vs Real\nR²={r2:.3f}', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature importance - CORREGIDO
top_10_features = feature_importance.head(10)
n_features = len(top_10_features)
colors_feat = plt.cm.viridis(np.linspace(0, 1, n_features))  # Mismo número que features

bars = axes[1].barh(range(n_features), top_10_features['importance'].values, 
                    color=colors_feat, edgecolor='black', linewidth=1.5)
axes[1].set_yticks(range(n_features))
axes[1].set_yticklabels(top_10_features['feature'].values, fontsize=10)
axes[1].set_xlabel('Importance', fontsize=12, fontweight='bold')
axes[1].set_title('Top Features Más Importantes', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('graficas_individuales/ml_prediccion_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Modelo de Machine Learning entrenado exitosamente")

---
# RESUMEN FINAL Y EXPORTACIÓN

In [ ]:
print("\n" + "="*70)
print("RESUMEN COMPLETO DEL ANÁLISIS")
print("="*70)

print("\n📊 DATASET:")
print(f"  Total de animes: {len(df):,}")
print(f"  Rango de años: {df_with_year['Released_Year'].min():.0f} - {df_with_year['Released_Year'].max():.0f}")
print(f"  Score promedio: {df['Score'].mean():.2f}")
print(f"  Géneros únicos: {len(all_genres)}")
print(f"  Estudios únicos: {len(studios_counts)}")

print("\n📁 ARCHIVOS GENERADOS:")
print("\n  GRÁFICAS INDIVIDUALES:")
individual_files = [
    '01_distribucion_scores.png',
    '02_top20_scores.png',
    '03_distribucion_tipo.png',
    '04_animes_por_año.png',
    '05_correlaciones.png',
    '06_top_generos.png',
    '07_top_estudios.png',
    '08_distribucion_temporada.png',
    'dashboard_completo.png',
    'seaborn_pairplot.png',
    'seaborn_jointplot.png',
    'seaborn_violinplots.png',
    'ml_prediccion_scores.png'
]
for i, f in enumerate(individual_files, 1):
    print(f"    {i:2d}. {f}")

print("\n  GRÁFICAS INTERACTIVAS:")
interactive_files = [
    'sistema_recomendacion_generos.html'
]
for i, f in enumerate(interactive_files, 1):
    print(f"    {i}. {f}")

print("\n  ANIMACIONES:")
animation_files = [
    'race_chart_generos.html',
    'evolucion_scores_generos.html',
    'scatter_evolutivo.html'
]
for i, f in enumerate(animation_files, 1):
    print(f"    {i}. {f}")

print("\n" + "="*70)
print("ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*70)
print("\n✅ Todos los archivos han sido generados")
print("✅ Dashboard completo con 10 gráficas")
print("✅ Análisis con Seaborn (pairplot, jointplot, violin)")
print("✅ 3 animaciones estilo Liga MX")
print("✅ Sistema de recomendación interactivo")
print("✅ Modelo de Machine Learning (bonus)")
print("\n🚀 Listo para presentar y deployment web!")

In [ ]:
# Guardar resumen estadístico
summary_stats = {
    'dataset': {
        'total_animes': int(len(df)),
        'score_mean': float(df['Score'].mean()),
        'score_median': float(df['Score'].median()),
        'year_range': f"{df_with_year['Released_Year'].min():.0f}-{df_with_year['Released_Year'].max():.0f}",
        'total_genres': len(all_genres),
        'total_studios': len(studios_counts)
    },
    'top_genres': [g for g, _ in genre_counts.most_common(10)],
    'top_studios': [s for s, _ in studios_counts.most_common(10)],
    'correlations': {
        'score_vs_members': float(corr_matrix.loc['Score', 'Members_num']),
        'score_vs_favorites': float(corr_matrix.loc['Score', 'Favorites_num'])
    },
    'files_generated': {
        'images': len(individual_files),
        'interactive': len(interactive_files),
        'animations': len(animation_files)
    }
}

with open('datos_procesados/resumen_analisis.json', 'w', encoding='utf-8') as f:
    json.dump(summary_stats, f, indent=4, ensure_ascii=False)

print("✓ Resumen guardado en: datos_procesados/resumen_analisis.json")